# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [61]:
orders_json = [
  {'Order':1,'Vendor_id':'V-01','lines':[{'Item':'Cheeseburger','qty':2},{'Item':'Soda','qty':1}]},
  {'Order':2,'Vendor_id':'V-10','lines':[{'Item':'Foam Finger','qty':1}]},
  {'Order':3,'Vendor_id':'V-07','lines':[{'Item':'Hot Dog','qty':3}]}
]
vendors_json = [
  {'Vendor_id':'V-01','Name':'Hoos Burgers','Zone':'A'},
  {'Vendor_id':'V-10','Name':'Cav Merch','Zone':'A'},
]

**One row of my result is one:** item

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [62]:
import pandas as pd

# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
orders = pd.json_normalize(orders_json, record_path='lines',meta=['Order', 'Vendor_id'])
print(orders)

# TODO: assert the row count and the quantity total
assert orders['qty'].sum() == 7
assert len(orders) == 4

           Item  qty Order Vendor_id
0  Cheeseburger    2     1      V-01
1          Soda    1     1      V-01
2   Foam Finger    1     2      V-10
3       Hot Dog    3     3      V-07


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [63]:
# TODO
vendors = pd.json_normalize(vendors_json)                 #flatten vendors_json
joined = orders.merge(vendors,on='Vendor_id',how='left')  #join vendors and orders
print(joined,'\n')
print('Unmatched vendor:', joined.loc[joined['Name'].isna(), 'Vendor_id'].to_list(),    #print the unmatched vendor id
      '| Units attached:', joined[joined['Vendor_id'] =='V-07']['qty'].to_list())       #print the units attached to unmatched vendor id

           Item  qty Order Vendor_id          Name Zone
0  Cheeseburger    2     1      V-01  Hoos Burgers    A
1          Soda    1     1      V-01  Hoos Burgers    A
2   Foam Finger    1     2      V-10     Cav Merch    A
3       Hot Dog    3     3      V-07           NaN  NaN 

Unmatched vendor: ['V-07'] | Units attached: [3]


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [64]:
# TODO
vendor_totals = joined.groupby('Name', dropna=False)['qty'].sum()
print(vendor_totals)
# Keep the unmatched vendor as NaN so its 3 units are not silently lost.

Name
Cav Merch       1
Hoos Burgers    3
NaN             3
Name: qty, dtype: int64


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [65]:
# TODO
zone_totals = joined.groupby('Zone', dropna=False)['qty'].sum().reset_index()   #create zone total sales dataframe
print(zone_totals)

print('\nHighest selling zone:', zone_totals.loc[zone_totals['qty'].idxmax(), 'Zone'], '|', 'Total:', #print highest total zone
      zone_totals.loc[zone_totals['qty'].idxmax(), 'qty'])
print('\nThe missing 3 units belong to V-07, which has no matching vendor and therefore no zone.')

  Zone  qty
0    A    4
1  NaN    3

Highest selling zone: A | Total: 4

The missing 3 units belong to V-07, which has no matching vendor and therefore no zone.


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [66]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
valid_orders = [x for x in ragged_json if 'lines' in x]                                       #determine orders that have lines
ragged = pd.json_normalize(valid_orders,record_path='lines',meta=['order', 'vendor_id'])      #create dataframe of ^

print(ragged)

    item  qty order vendor_id
0   Soda  2.0     4      V-01
1  Fries  NaN     6      V-01


### Q6 — Validate

In [70]:
assert len(orders) == 4
assert orders['qty'].sum() == 7
assert len(joined) == len(orders), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

__a)__
One row per line item is useful because each individual item and its quantity can be analyzed directly. For example, it becomes easy to calculate the total quantity sold for each item, such as how many Sodas or Hot Dogs were sold. If there were one row per order, multiple line items would be stored inside nested lists, making item-level analysis much harder.

__b)__
A missing quantity means the quantity was not provided or is unknown, while a quantity of zero means the quantity was explicitly recorded as zero. If we replace missing quantities with zero, we incorrectly treat unknown data as confirmed zero sales. This can cause incorrect totals, averages, inventory calculations, and business decisions downstream.